# Détection de `coordination_risk` dans les topics Gerrit

La cible `coordination_risk` est déjà calculée dans `change_level_dataset_coordination_risk.csv`.
Le notebook charge ce jeu étiqueté directement; il ne reconstruit pas les labels à chaque exécution.
La règle enregistrée dans `coordination_risk_label_metadata.json` est :

```text
coordination_risk = 1 si
    (graph_in_degree >= P75 AND first_review_delay_hours >= P75
     AND time_open_hours >= P75)
    ET status != MERGED
    = 0 sinon
```

Les trois seuils P75 ont été calculés une fois sur la partition de fit, avec les mêmes
séparations par topic que celles utilisées ci-dessous. Les valeurs exactes sont conservées
dans le fichier de métadonnées. Le script `build_coordination_risk_labels.py` permet de
reconstruire le CSV si les données sources changent.

Les colonnes qui définissent la cible sont exclues des variables d'entrée. Le suréchantillonnage
concerne uniquement le fit; le seuil de classification est choisi sur la validation et
l'évaluation finale utilise le test intact.


## 1. Imports

In [1]:
import os
os.environ.setdefault("LOKY_MAX_CPU_COUNT", str(os.cpu_count() or 1))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    classification_report, average_precision_score, roc_auc_score,
    confusion_matrix, f1_score, precision_score, recall_score, precision_recall_curve,
)

pd.set_option('display.max_columns', 50)
RANDOM_STATE = 42


Could not save font_manager cache [Errno 13] Permission denied: 'C:\\Users\\ziedd\\AppData\\Local\\matplotlib\\fontlist-v3.11.0.json.matplotlib-lock'


## 2. Chargement

In [2]:
INPUT_CSV = "change_level_dataset_coordination_risk.csv"
df = pd.read_csv(INPUT_CSV, low_memory=False)
print(f"Dataset: {df.shape[0]:,} rows, {df.shape[1]} columns")
print(df["coordination_risk"].value_counts().rename(index={0: "not_at_risk", 1: "coordination_risk"}))
df.head()


Dataset: 83,500 rows, 27 columns
coordination_risk
not_at_risk          70477
coordination_risk    13023
Name: count, dtype: int64


In [3]:
df.isna().sum().sort_values(ascending=False)


## 3. Nettoyage

In [4]:
for col in ['work_in_progress', 'mergeable']:
    if df[col].dtype == object:
        df[col] = df[col].map({'True': 1, 'False': 0, True: 1, False: 0})
    df[col] = df[col].fillna(0).astype(int)

# Réduction de cardinalité pour 'project'
TOP_N_PROJECTS = 30
top_projects = df['project'].value_counts().nlargest(TOP_N_PROJECTS).index
df['project'] = df['project'].where(df['project'].isin(top_projects), other='OTHER')

# Valeurs manquantes sur les colonnes utilisées pour la définition du target : NaN -> 0.
# Un délai ou une durée non mesuré(e) ne satisfait donc pas la condition >= P75.
df['first_review_delay_hours'] = df['first_review_delay_hours'].fillna(0)
df['time_open_hours'] = df['time_open_hours'].fillna(0)
df['graph_in_degree'] = df['graph_in_degree'].fillna(0)

df[['project']].value_counts().head(10)


## 4. Cible pré-calculée `coordination_risk`

Les labels sont lus depuis le CSV préparé. Cette cellule ne recalcule ni les seuils ni les labels.


## 5. Séparation entraînement, validation et test par topic

Les changements d'un même topic restent dans une seule partition. Les lignes sans topic
sont traitées comme des groupes individuels.


In [5]:
def topic_aware_split(df, test_size=0.2, random_state=RANDOM_STATE):
    df = df.copy()
    no_topic_mask = df['topic'].isna()
    df.loc[no_topic_mask, '_split_group'] = [f"__no_topic_{i}" for i in df.index[no_topic_mask]]
    df.loc[~no_topic_mask, '_split_group'] = df.loc[~no_topic_mask, 'topic']

    gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
    train_idx, test_idx = next(gss.split(df, groups=df['_split_group']))
    return (
        df.iloc[train_idx].drop(columns=['_split_group']).copy(),
        df.iloc[test_idx].drop(columns=['_split_group']).copy(),
    )


train_pool_df, test_df = topic_aware_split(df, test_size=0.2, random_state=RANDOM_STATE)
fit_df, val_df = topic_aware_split(train_pool_df, test_size=0.2, random_state=137)
print(f"Fit : {len(fit_df)} lignes | Validation : {len(val_df)} lignes | Test : {len(test_df)} lignes")


Fit : 52122 lignes | Validation : 14582 lignes | Test : 16796 lignes


In [6]:
# Contrôle du taux de labels dans chaque partition
for split_name, split_df in [("Fit", fit_df), ("Validation", val_df), ("Test", test_df)]:
    print(f"Taux coordination_risk {split_name.lower()} : {split_df['coordination_risk'].mean():.2%}")


Taux coordination_risk fit : 15.86%
Taux coordination_risk validation : 14.59%
Taux coordination_risk test : 15.65%


## 6. Définition des features (X) — exclusion explicite des colonnes de fuite

In [7]:
LEAKAGE_OR_ID_COLS = [
    'number', 'change_id',
    'graph_in_degree',                  # utilisé pour définir le target
    'first_review_delay_hours',         # utilisé pour définir le target
    'time_open_hours',                  # utilisé pour définir le target
    'status',                           # utilisé pour définir le target
    'graph_out_degree',                 # exclu pour cohérence avec les variables du graphe
    'coordination_risk',
]
DATE_COLS = ['created', 'updated', 'submitted']
ID_LIKE_COLS = ['topic', 'node_source', 'project', 'branch']
CATEGORICAL_COLS = []  # 'status' retire (utilise pour le target)

drop_cols = set(LEAKAGE_OR_ID_COLS + DATE_COLS + ID_LIKE_COLS)
feature_cols = [c for c in df.columns if c not in drop_cols]
numeric_cols = [c for c in feature_cols if c not in CATEGORICAL_COLS]

print(f"{len(feature_cols)} features : {feature_cols}")


12 features : ['work_in_progress', 'mergeable', 'insertions', 'deletions', 'total_comment_count', 'unresolved_comment_count', 'total_patchsets', 'n_files_changed', 'total_messages', 'n_reviewers', 'avg_time_between_patchsets_hours', 'ci_messages_count']


In [8]:
X_fit, y_fit = fit_df[feature_cols], fit_df['coordination_risk']
X_val, y_val = val_df[feature_cols], val_df['coordination_risk']
X_test, y_test = test_df[feature_cols], test_df['coordination_risk']


## 7. Preprocessing

In [9]:
numeric_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale', StandardScaler()),
])
categorical_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='constant', fill_value='UNKNOWN')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipe, numeric_cols),
])


## 8. Entraînement avec suréchantillonnage limité au fit

Les positifs sont suréchantillonnés aléatoirement pour représenter 20 % des exemples
utilisés à l'entraînement. La validation et le test ne sont pas rééquilibrés.


In [10]:
def oversample_to_positive_rate(X, y, target_rate=0.20, random_state=RANDOM_STATE):
    """Oversample positive fit rows only; do not use on validation or test data."""
    fit_data = X.copy()
    fit_data["_target"] = np.asarray(y)
    positives = fit_data[fit_data["_target"] == 1]
    negatives = fit_data[fit_data["_target"] == 0]
    target_positive_count = int(np.ceil(len(negatives) * target_rate / (1 - target_rate)))
    target_positive_count = max(target_positive_count, len(positives))
    sampled_positives = positives.sample(
        n=target_positive_count,
        replace=target_positive_count > len(positives),
        random_state=random_state,
    )
    balanced = pd.concat([negatives, sampled_positives]).sample(
        frac=1, random_state=random_state
    )
    return balanced[feature_cols], balanced["_target"].astype(int)


def select_threshold_on_validation(y_true, y_proba):
    precisions, recalls, thresholds = precision_recall_curve(y_true, y_proba)
    f1_values = 2 * precisions[:-1] * recalls[:-1] / (precisions[:-1] + recalls[:-1] + 1e-9)
    best_idx = int(np.argmax(f1_values))
    return float(thresholds[best_idx]), float(f1_values[best_idx])


def evaluate(name, y_true, y_pred, y_proba):
    print(f"\n{'='*60}\n{name}\n{'='*60}")
    print(classification_report(y_true, y_pred, target_names=['not_at_risk', 'coordination_risk'], zero_division=0))
    print(f"Average Precision (PR-AUC) : {average_precision_score(y_true, y_proba):.4f}")
    print(f"ROC-AUC                    : {roc_auc_score(y_true, y_proba):.4f}")
    print(f"F1 (coordination_risk)     : {f1_score(y_true, y_pred, zero_division=0):.4f}")
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    print(f"Confusion matrix -> TN={tn}  FP={fp}  FN={fn}  TP={tp}")


models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    'Random Forest': RandomForestClassifier(
        n_estimators=300, min_samples_leaf=5,
        random_state=RANDOM_STATE, n_jobs=-1,
    ),
    'HistGradientBoosting': HistGradientBoostingClassifier(random_state=RANDOM_STATE),
}

X_fit_balanced, y_fit_balanced = oversample_to_positive_rate(X_fit, y_fit, target_rate=0.20)
print(f"Fit positives before/after oversampling: {y_fit.sum()} / {y_fit_balanced.sum()}")
print(f"Positive rate after oversampling: {y_fit_balanced.mean():.2%}")

fitted_pipelines = {}
threshold_results = {}
for name, model in models.items():
    pipe = Pipeline([('prep', preprocessor), ('clf', clone(model))])
    pipe.fit(X_fit_balanced, y_fit_balanced)
    fitted_pipelines[name] = pipe

    val_proba = pipe.predict_proba(X_val)[:, 1]
    chosen_threshold, validation_f1 = select_threshold_on_validation(y_val, val_proba)
    test_proba = pipe.predict_proba(X_test)[:, 1]
    test_pred = (test_proba >= chosen_threshold).astype(int)
    default_pred = (test_proba >= 0.5).astype(int)

    test_precision = precision_score(y_test, test_pred, zero_division=0)
    test_recall = recall_score(y_test, test_pred, zero_division=0)
    test_f1 = f1_score(y_test, test_pred, zero_division=0)
    threshold_results[name] = {
        'threshold': chosen_threshold,
        'validation_f1': validation_f1,
        'precision': test_precision,
        'recall': test_recall,
        'f1': test_f1,
        'default_f1': f1_score(y_test, default_pred, zero_division=0),
        'proba': test_proba,
    }

    print(f"\n{name}: validation threshold={chosen_threshold:.4f}; validation F1={validation_f1:.4f}")
    print(f"Test F1 at default threshold 0.5: {threshold_results[name]['default_f1']:.4f}")
    print("Test metrics at validation-selected threshold:")
    evaluate(name, y_test, test_pred, test_proba)


Fit positives before/after oversampling: 8267 / 10964
Positive rate after oversampling: 20.00%

Logistic Regression: validation threshold=0.1950; validation F1=0.4169
Test F1 at default threshold 0.5: 0.1934
Test metrics at validation-selected threshold:

Logistic Regression
                   precision    recall  f1-score   support

      not_at_risk       0.90      0.80      0.85     14167
coordination_risk       0.32      0.52      0.40      2629

         accuracy                           0.75     16796
        macro avg       0.61      0.66      0.62     16796
     weighted avg       0.81      0.75      0.78     16796

Average Precision (PR-AUC) : 0.3198
ROC-AUC                    : 0.6794
F1 (coordination_risk)     : 0.4009
Confusion matrix -> TN=11291  FP=2876  FN=1249  TP=1380



Random Forest: validation threshold=0.3205; validation F1=0.6287
Test F1 at default threshold 0.5: 0.5707
Test metrics at validation-selected threshold:

Random Forest
                   precision    recall  f1-score   support

      not_at_risk       0.94      0.90      0.92     14167
coordination_risk       0.56      0.67      0.61      2629

         accuracy                           0.86     16796
        macro avg       0.75      0.78      0.76     16796
     weighted avg       0.88      0.86      0.87     16796

Average Precision (PR-AUC) : 0.5951
ROC-AUC                    : 0.8688
F1 (coordination_risk)     : 0.6067
Confusion matrix -> TN=12766  FP=1401  FN=874  TP=1755



HistGradientBoosting: validation threshold=0.3373; validation F1=0.6350
Test F1 at default threshold 0.5: 0.5872
Test metrics at validation-selected threshold:

HistGradientBoosting
                   precision    recall  f1-score   support

      not_at_risk       0.94      0.90      0.92     14167
coordination_risk       0.56      0.67      0.61      2629

         accuracy                           0.86     16796
        macro avg       0.75      0.78      0.76     16796
     weighted avg       0.88      0.86      0.87     16796

Average Precision (PR-AUC) : 0.6168
ROC-AUC                    : 0.8723
F1 (coordination_risk)     : 0.6066
Confusion matrix -> TN=12776  FP=1391  FN=879  TP=1750


## 9. Courbes precision-recall

In [11]:
fig, ax = plt.subplots(figsize=(6, 5))
for name, pipe in fitted_pipelines.items():
    y_proba = pipe.predict_proba(X_test)[:, 1]
    precision, recall, _ = precision_recall_curve(y_test, y_proba)
    ax.plot(recall, precision, label=name)

ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall curve — coordination_risk')
ax.legend()
plt.show()


## 10. Feature importance (Random Forest)

In [12]:
rf_pipe = fitted_pipelines['Random Forest']
all_feature_names = numeric_cols
importances = rf_pipe.named_steps['clf'].feature_importances_
imp_df = pd.DataFrame({'feature': all_feature_names, 'importance': importances})
imp_df = imp_df.sort_values('importance', ascending=False).head(20)

imp_df.plot(kind='barh', x='feature', y='importance', figsize=(8, 8), legend=False)
plt.gca().invert_yaxis()
plt.title('Top 20 features (Random Forest)')
plt.tight_layout()
plt.show()

imp_df


## 11. Seuil de décision choisi sur la validation

Le seuil qui maximise le F1 est choisi avec les probabilités de validation uniquement.
Les métriques de test ci-dessous sont calculées avec ce seuil fixe; le test n'est pas
utilisé pour régler le seuil.


In [13]:
threshold_summary = pd.DataFrame([
    {
        "model": name,
        "threshold_selected_on_validation": res["threshold"],
        "validation_f1": res["validation_f1"],
        "test_f1_at_0.5": res["default_f1"],
        "test_precision": res["precision"],
        "test_recall": res["recall"],
        "test_f1_at_validation_threshold": res["f1"],
        "test_average_precision": average_precision_score(y_test, res["proba"]),
    }
    for name, res in threshold_results.items()
]).sort_values("test_average_precision", ascending=False)
threshold_summary


## 12. Validation croisée GroupKFold par topic

La validation croisée utilise les labels coordination_risk pré-calculés et conservés
dans le jeu de données. Le suréchantillonnage est appliqué uniquement au sous-ensemble
d'entraînement de chaque fold. L'Average Precision est calculée sur le fold de
validation avec sa distribution naturelle.


In [14]:
def make_cv_groups(d):
    d = d.copy()
    no_topic_mask = d['topic'].isna()
    d.loc[no_topic_mask, '_cv_group'] = [f"__no_topic_{i}" for i in d.index[no_topic_mask]]
    d.loc[~no_topic_mask, '_cv_group'] = d.loc[~no_topic_mask, 'topic']
    return d['_cv_group']


cv_groups = make_cv_groups(df)
gkf = GroupKFold(n_splits=5)
cv_results = {name: [] for name in models}

for fold, (train_idx, val_idx) in enumerate(gkf.split(df, groups=cv_groups), 1):
    fold_fit = df.iloc[train_idx].copy()
    fold_val = df.iloc[val_idx].copy()
    y_fold_fit = fold_fit["coordination_risk"]
    y_fold_val = fold_val["coordination_risk"]

    X_fold_fit, y_fold_fit = oversample_to_positive_rate(
        fold_fit[feature_cols], y_fold_fit, target_rate=0.20,
        random_state=RANDOM_STATE + fold,
    )
    X_fold_val = fold_val[feature_cols]

    for name, model in models.items():
        pipe = Pipeline([('prep', clone(preprocessor)), ('clf', clone(model))])
        pipe.fit(X_fold_fit, y_fold_fit)
        proba = pipe.predict_proba(X_fold_val)[:, 1]
        cv_results[name].append(average_precision_score(y_fold_val, proba))

print("Average Precision moyen +/- écart-type sur 5 folds GroupKFold par topic :\n")
for name, scores in cv_results.items():
    print(f"  {name:<22s} : {np.mean(scores):.4f} +/- {np.std(scores):.4f}   "
          f"{['%.3f' % s for s in scores]}")


Average Precision moyen +/- écart-type sur 5 folds GroupKFold par topic :

  Logistic Regression    : 0.3628 +/- 0.0115   ['0.363', '0.354', '0.354', '0.358', '0.385']
  Random Forest          : 0.6368 +/- 0.0166   ['0.637', '0.618', '0.620', '0.647', '0.662']
  HistGradientBoosting   : 0.6482 +/- 0.0099   ['0.645', '0.634', '0.646', '0.651', '0.665']


## 13. Comparaison finale

In [15]:
results_summary = []
for name, res in threshold_results.items():
    results_summary.append({
        "model": name,
        "threshold_selected_on_validation": round(res["threshold"], 4),
        "validation_f1": round(res["validation_f1"], 4),
        "test_precision": round(res["precision"], 4),
        "test_recall": round(res["recall"], 4),
        "test_f1": round(res["f1"], 4),
        "test_average_precision": round(average_precision_score(y_test, res["proba"]), 4),
        "cv_average_precision_mean": round(np.mean(cv_results[name]), 4),
        "cv_average_precision_std": round(np.std(cv_results[name]), 4),
    })

summary_df = pd.DataFrame(results_summary).sort_values("cv_average_precision_mean", ascending=False)
summary_df


## 14. Limites à mentionner dans le rapport de stage

- **Détection plutôt que prédiction précoce** : les délais de revue et la durée
  d'ouverture participent à la définition de la cible. Le modèle détecte des
  signaux déjà observés; il ne prédit pas nécessairement un futur goulot
  avant leur apparition.

- **Définition de la cible** : un change non fusionné est étiqueté positif
  uniquement lorsque les trois signaux — graph_in_degree,
  first_review_delay_hours et time_open_hours — atteignent simultanément
  leur P75. La règle et les seuils exacts sont enregistrés dans
  coordination_risk_label_metadata.json.

- **Suréchantillonnage** : les exemples positifs sont dupliqués uniquement dans
  les partitions d'entraînement. Les métriques de validation et de test
  gardent la distribution naturelle.

- **Seuil de classification** : le seuil maximisant le F1 est choisi sur la
  validation; les résultats du test sont ensuite calculés avec ce seuil fixe.
